In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import category_encoders as ce

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.preprocessing import LabelEncoder,StandardScaler,Normalizer
from sklearn.metrics import f1_score,r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
import lightgbm as lgb
#import xgboost as xgb

import seaborn as sns
import matplotlib.pyplot as plt


import warnings
warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv('../data/favorita-grocery-sales-forecasting/train.csv')


,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


In [23]:

# 2. Сохраняем данные в формат Parquet
data.to_parquet('output.parquet', engine='pyarrow')

data.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


In [24]:
data['date'] = pd.to_datetime(data['date'])

In [25]:
data_last_year = data[data['date'] >= '2016-08-15'].copy()

In [9]:
data_last_year.dtypes

id                      int64
date           datetime64[us]
store_nbr               int64
item_nbr                int64
unit_sales            float64
onpromotion            object
dtype: object

In [26]:
x_test = pd.read_csv('../data/favorita-grocery-sales-forecasting/test.csv')
x_test.to_parquet('output.parquet', engine='pyarrow')

x_test.dtypes

id             int64
date             str
store_nbr      int64
item_nbr       int64
onpromotion     bool
dtype: object

In [27]:
# 1. СКЛЕИВАЕМ (test прилипает к концу train)
all_data = pd.concat([data_last_year, x_test], ignore_index=True)
all_data['date'] = pd.to_datetime(all_data['date'])
# 2. СОРТИРУЕМ общую таблицу (теперь тестовые дни встанут ровно после тренировочных для каждого товара)
all_data = all_data.sort_values(by=['store_nbr', 'item_nbr', 'date'])

# 3. СЧИТАЕМ ЛАГИ (безопасные для 16 дней теста)
all_data['unit_sales_lag_1'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(21)
all_data['unit_sales_lag_7'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(28)

# Среднее за 2 дня (с безопасным отступом в 16 дней)
all_data['unit_sales_mean_2'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'] \
    .transform(lambda x: x.shift(16).rolling(window=2).mean())

# Среднее за 4 дня
all_data['unit_sales_mean_4'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'] \
    .transform(lambda x: x.shift(16).rolling(window=4).mean())

# Среднее за 6 дней
all_data['unit_sales_mean_6'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'] \
    .transform(lambda x: x.shift(16).rolling(window=6).mean())

all_data['unit_sales_std_2'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'] \
    .transform(lambda x: x.shift(16).rolling(window=2).std())


all_data['unit_sales_std_4'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'] \
    .transform(lambda x: x.shift(16).rolling(window=4).std())


all_data['unit_sales_std_6'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'] \
    .transform(lambda x: x.shift(16).rolling(window=6).std())


In [ ]:
# --- 1. ОПТИМИЗАЦИЯ ПАМЯТИ ПЕРЕД РАСЧЕТАМИ ---
# Уменьшаем вес колонки продаж до 32-битного формата (экономит массу RAM)
all_data['unit_sales'] = all_data['unit_sales'].astype('float32')

# Убеждаемся, что данные отсортированы
all_data = all_data.sort_values(by=['store_nbr', 'item_nbr', 'date'])

# --- 2. БЫСТРЫЕ РАСЧЕТЫ (без дублирования) ---
# Шаг А: Считаем базовый сдвиг ОДИН РАЗ (это очень быстрая операция)
all_data['lag_16'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(16)

# Также ужимаем вес новой колонки
all_data['lag_16'] = all_data['lag_16'].astype('float32')

# Шаг Б: Считаем скользящие средние поверх УЖЕ сдвинутой колонки
all_data['mean_2'] = all_data.groupby(['store_nbr', 'item_nbr'])['lag_16'].transform(lambda x: x.rolling(2).mean()).astype('float32')
all_data['mean_4'] = all_data.groupby(['store_nbr', 'item_nbr'])['lag_16'].transform(lambda x: x.rolling(4).mean()).astype('float32')
all_data['mean_6'] = all_data.groupby(['store_nbr', 'item_nbr'])['lag_16'].transform(lambda x: x.rolling(6).mean()).astype('float32')

# Дополнительные лаги
all_data['lag_21'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(21).astype('float32')
all_data['lag_28'] = all_data.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(28).astype('float32')

In [28]:

# 4. РАЗДЕЛЯЕМ ОБРАТНО
# Тренировочные данные — те, где изначально были продажи
train_features = all_data[all_data['unit_sales'].notna()].copy()

# Тестовые данные — те, которые мы должны предсказать
test_after_split = all_data[all_data['unit_sales'].isna()].copy()

# 3. Удаляем строки с NaN, которые образовались в начале train из-за лагов и окон
# (В test_features мы NaN не удаляем, там должно остаться ровно столько строк, сколько ждет Kaggle!)
train_features = train_features.dropna()

In [11]:
print(train_features.dtypes)
test_after_split.dtypes


id                           int64
date                datetime64[us]
store_nbr                    int64
item_nbr                     int64
unit_sales                 float64
onpromotion                 object
unit_sales_lag_1           float64
unit_sales_lag_7           float64
dtype: object


id                           int64
date                datetime64[us]
store_nbr                    int64
item_nbr                     int64
unit_sales                 float64
onpromotion                 object
unit_sales_lag_1           float64
unit_sales_lag_7           float64
dtype: object

In [29]:

y = train_features.unit_sales
X = train_features.drop('unit_sales',axis=1)
#print(X['date'].max())



# Извлекаем полезные числовые признаки из даты
X['year'] = X['date'].dt.year
X['month'] = X['date'].dt.month
X['day'] = X['date'].dt.day

# День недели (от 0 до 6) — критически важный признак для прогнозирования продаж!
X['day_of_week'] = X['date'].dt.dayofweek 


In [30]:
for col in X.select_dtypes(include=['float64']).columns:
    X[col] = X[col].astype('float32')

for col in X.select_dtypes(include=['int64']).columns:
    X[col] = X[col].astype('int32')

In [31]:
test_ids = test_after_split['id']

# 2. Удаляем id из тестовой выборки (модель должна видеть только признаки!)
x_test_features = test_after_split.drop('id', axis=1)
x_test_features['date'] = pd.to_datetime(x_test_features['date'])

# Извлекаем полезные числовые признаки из даты
x_test_features['year'] = x_test_features['date'].dt.year
x_test_features['month'] = x_test_features['date'].dt.month
x_test_features['day'] = x_test_features['date'].dt.day

# День недели (от 0 до 6) — критически важный признак для прогнозирования продаж!
x_test_features['day_of_week'] = x_test_features['date'].dt.dayofweek 


# Удаляем оригинальный столбец с датой, чтобы он не ломал модель
x_test_features = x_test_features.drop('date', axis=1)

x_test_features['onpromotion'] = x_test_features['onpromotion'].map({True: 1, False: 0}).fillna(-1).astype(int)

x_test_features = x_test_features.drop(['unit_sales'], axis=1)

In [15]:
x_test_features.dtypes
y.shape
x_test_features.shape

(3370464, 9)

In [32]:
# Удаляем оригинальный столбец с датой, чтобы он не ломал модель
X = X.drop('date', axis=1)

train_ids = X['id']

# 2. Удаляем id из тестовой выборки (модель должна видеть только признаки!)
x_train_features = X.drop('id', axis=1)


In [33]:

x_train_features['onpromotion'] = x_train_features['onpromotion'].map({True: 1, False: 0}).fillna(-1).astype(int)


In [18]:

print(x_train_features.shape)
print(x_train_features.dtypes)
print(x_test_features.shape)
x_test_features.dtypes


(37454835, 9)
store_nbr             int32
item_nbr              int32
onpromotion           int64
unit_sales_lag_1    float32
unit_sales_lag_7    float32
year                  int32
month                 int32
day                   int32
day_of_week           int32
dtype: object
(3370464, 9)


store_nbr             int64
item_nbr              int64
onpromotion           int64
unit_sales_lag_1    float64
unit_sales_lag_7    float64
year                  int32
month                 int32
day                   int32
day_of_week           int32
dtype: object

In [34]:

#model = lgb.LGBMRegressor(learning_rate=0.1,n_estimators=100,max_depth=5,num_leaves=50)
model = lgb.LGBMRegressor(n_estimators=300,max_depth=10,random_state=42,n_jobs=-1)
model.fit(x_train_features,y)
predictions = model.predict(x_test_features)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.663852 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2407
[LightGBM] [Info] Number of data points in the train set: 32815810, number of used features: 15
[LightGBM] [Info] Start training from score 8.416200


In [35]:
submission = pd.DataFrame({
    'id': test_ids,         # Подставляем сохраненные номера из пункта 1
    'unit_sales': predictions    # Подставляем результат из пункта 3
})
submission['unit_sales'] = submission['unit_sales'].clip(lower=0)
# 5. Сохраняем в файл строго без внутренних индексов Pandas
submission.to_csv('submission5.csv', index=False)

In [ ]:
data_oil = pd.read_csv('../data/favorita-grocery-sales-forecasting/oil.csv')
data_oil.to_parquet('output.parquet', engine='pyarrow')
data_oil.dtypes